# Lab 06.1: Ray Serve for LLM Inference

Deploy LLM inference with Ray Serve: deployment classes, autoscaling profiles,
prefix-aware routing, load testing, and HAProxy integration.

In [ ]:
import sys
sys.path.insert(0, '../../..')

import time
import asyncio
import numpy as np
from dataclasses import dataclass
from typing import Dict, List
from content.utils.benchmark import BenchmarkTimer
from content.utils.latency import LatencyTracker

## 1. Ray Serve Deployment Class

A deployment class wraps model loading and inference behind a scalable HTTP endpoint.

In [ ]:
class LLMDeployment:
    """Ray Serve deployment for LLM inference."""

    def __init__(self, model_id: str = "meta-llama/Llama-2-7b-hf", max_batch_size: int = 8):
        self.model_id = model_id
        self.max_batch_size = max_batch_size
        self.request_count = 0
        self._kv_cache: Dict[str, List[float]] = {}  # prefix -> cached activations
        print(f"Initialized deployment: {model_id}, batch_size={max_batch_size}")

    async def generate(self, prompt: str, max_tokens: int = 128) -> Dict:
        self.request_count += 1
        start = time.perf_counter()
        # Simulate TTFT + token generation
        prefix_hit = any(prompt.startswith(p) for p in self._kv_cache)
        ttft = 0.02 if prefix_hit else 0.15
        await asyncio.sleep(ttft + max_tokens * 0.008)
        latency = time.perf_counter() - start
        # Cache prefix
        prefix_key = prompt[:64]
        self._kv_cache[prefix_key] = [0.0] * 32
        return {
            "text": f"[generated {max_tokens} tokens]",
            "latency_s": round(latency, 4),
            "prefix_cache_hit": prefix_hit,
            "request_id": self.request_count
        }

# Test deployment
deploy = LLMDeployment()
result = asyncio.get_event_loop().run_until_complete(deploy.generate("Explain quantum computing"))
print(f"First request: {result}")
result2 = asyncio.get_event_loop().run_until_complete(deploy.generate("Explain quantum computing in detail"))
print(f"Prefix-cached request: {result2}")

## 2. Autoscaling Profiles

Three profiles for different workload patterns: conservative, balanced, aggressive.

In [ ]:
@dataclass
class AutoscalingProfile:
    name: str
    min_replicas: int
    max_replicas: int
    target_ongoing_requests: int
    upscale_delay_s: float
    downscale_delay_s: float
    smoothing_factor: float  # EMA for load signal

PROFILES = {
    "conservative": AutoscalingProfile(
        name="conservative", min_replicas=2, max_replicas=8,
        target_ongoing_requests=3, upscale_delay_s=60.0,
        downscale_delay_s=300.0, smoothing_factor=0.3
    ),
    "balanced": AutoscalingProfile(
        name="balanced", min_replicas=1, max_replicas=16,
        target_ongoing_requests=5, upscale_delay_s=15.0,
        downscale_delay_s=120.0, smoothing_factor=0.5
    ),
    "aggressive": AutoscalingProfile(
        name="aggressive", min_replicas=1, max_replicas=32,
        target_ongoing_requests=8, upscale_delay_s=5.0,
        downscale_delay_s=60.0, smoothing_factor=0.7
    ),
}

for name, p in PROFILES.items():
    print(f"{name:>12}: replicas=[{p.min_replicas},{p.max_replicas}], "
          f"target_qps/replica={p.target_ongoing_requests}, "
          f"upscale={p.upscale_delay_s}s, downscale={p.downscale_delay_s}s")

In [ ]:
class AutoscalerSimulator:
    """Simulates replica scaling decisions over time."""

    def __init__(self, profile: AutoscalingProfile):
        self.profile = profile
        self.current_replicas = profile.min_replicas
        self.smoothed_load = 0.0
        self.last_scale_time = 0.0

    def step(self, ongoing_requests: int, current_time: float) -> int:
        p = self.profile
        self.smoothed_load = (p.smoothing_factor * ongoing_requests +
                              (1 - p.smoothing_factor) * self.smoothed_load)
        desired = max(p.min_replicas, min(p.max_replicas,
                      int(np.ceil(self.smoothed_load / p.target_ongoing_requests))))

        time_since_scale = current_time - self.last_scale_time
        if desired > self.current_replicas and time_since_scale >= p.upscale_delay_s:
            self.current_replicas = desired
            self.last_scale_time = current_time
        elif desired < self.current_replicas and time_since_scale >= p.downscale_delay_s:
            self.current_replicas = desired
            self.last_scale_time = current_time
        return self.current_replicas

# Simulate a traffic spike
traffic = [2]*10 + [20]*20 + [50]*15 + [10]*20 + [2]*15  # 80 steps
print(f"{'Profile':<14} {'Peak Replicas':>14} {'Avg Replicas':>13} {'Scale Events':>13}")
print("-" * 58)
for name, profile in PROFILES.items():
    sim = AutoscalerSimulator(profile)
    replicas_history = []
    for t, load in enumerate(traffic):
        r = sim.step(load, t * 5.0)  # 5s intervals
        replicas_history.append(r)
    changes = sum(1 for i in range(1, len(replicas_history)) if replicas_history[i] != replicas_history[i-1])
    print(f"{name:<14} {max(replicas_history):>14} {np.mean(replicas_history):>13.1f} {changes:>13}")

## 3. Prefix-Aware Routing

Route requests to replicas that already have the relevant KV cache prefix loaded,
reducing TTFT by avoiding redundant prefill computation.

In [ ]:
class PrefixAwareRouter:
    """Routes requests to replicas with matching cached prefixes."""

    def __init__(self, num_replicas: int = 4):
        self.num_replicas = num_replicas
        # Each replica tracks its cached prefixes
        self.replica_prefixes: Dict[int, set] = {i: set() for i in range(num_replicas)}
        self.replica_load: Dict[int, int] = {i: 0 for i in range(num_replicas)}

    def route(self, prompt: str) -> int:
        prefix = prompt[:64]
        # Find replicas with matching prefix
        matches = [r for r, prefixes in self.replica_prefixes.items() if prefix in prefixes]
        if matches:
            # Pick least-loaded among matches
            target = min(matches, key=lambda r: self.replica_load[r])
        else:
            # Least-loaded overall
            target = min(range(self.num_replicas), key=lambda r: self.replica_load[r])
            self.replica_prefixes[target].add(prefix)
        self.replica_load[target] += 1
        return target

    def release(self, replica_id: int):
        self.replica_load[replica_id] = max(0, self.replica_load[replica_id] - 1)

# Simulate routing with shared-prefix workload
router = PrefixAwareRouter(num_replicas=4)
prompts = [
    "You are a helpful assistant. Explain" + f" topic {i}" for i in range(20)
] + [f"Summarize the following document: doc_{i}" for i in range(10)]

routing_decisions = []
for prompt in prompts:
    replica = router.route(prompt)
    routing_decisions.append(replica)
    router.release(replica)

# Measure prefix locality
from collections import Counter
dist = Counter(routing_decisions)
print("Routing distribution across replicas:")
for r in range(4):
    print(f"  Replica {r}: {dist[r]} requests, {len(router.replica_prefixes[r])} cached prefixes")
print(f"\nPrefix locality score: {max(dist.values()) / sum(dist.values()):.1%} "
      f"(higher = better prefix reuse)")

In [ ]:
# Compare prefix-aware vs round-robin routing latency
async def simulate_routing_latency(router_type: str, prompts: List[str]) -> Dict:
    latencies = []
    cache_hits = 0
    seen_prefixes = set()

    for i, prompt in enumerate(prompts):
        prefix = prompt[:64]
        if router_type == "prefix_aware":
            hit = prefix in seen_prefixes
        else:  # round-robin: no affinity, cache miss every time
            hit = False
        seen_prefixes.add(prefix)
        ttft = 0.02 if hit else 0.15
        latencies.append(ttft + 128 * 0.008)
        if hit:
            cache_hits += 1

    return {
        "router": router_type,
        "p50_ms": np.percentile(latencies, 50) * 1000,
        "p99_ms": np.percentile(latencies, 99) * 1000,
        "cache_hit_rate": cache_hits / len(prompts)
    }

# Repeated prompts with shared system prefix
test_prompts = ["You are a helpful assistant. " + f"Question {i % 5}" for i in range(100)]
rr = asyncio.get_event_loop().run_until_complete(simulate_routing_latency("round_robin", test_prompts))
pa = asyncio.get_event_loop().run_until_complete(simulate_routing_latency("prefix_aware", test_prompts))

print(f"{'Router':<14} {'P50 (ms)':>10} {'P99 (ms)':>10} {'Cache Hit':>10}")
print("-" * 48)
for r in [rr, pa]:
    print(f"{r['router']:<14} {r['p50_ms']:>10.1f} {r['p99_ms']:>10.1f} {r['cache_hit_rate']:>10.1%}")

## 4. Load Testing Client

Concurrent load generator measuring throughput, latency percentiles, and error rates.

In [ ]:
class LoadTestClient:
    """Async load tester for LLM serving endpoints."""

    def __init__(self, deployment: LLMDeployment, concurrency: int = 10):
        self.deployment = deployment
        self.concurrency = concurrency
        self.latencies: List[float] = []
        self.errors = 0

    async def _send_request(self, prompt: str, sem: asyncio.Semaphore):
        async with sem:
            try:
                result = await self.deployment.generate(prompt, max_tokens=64)
                self.latencies.append(result["latency_s"])
            except Exception:
                self.errors += 1

    async def run(self, prompts: List[str]) -> Dict:
        self.latencies = []
        self.errors = 0
        sem = asyncio.Semaphore(self.concurrency)
        start = time.perf_counter()
        await asyncio.gather(*[self._send_request(p, sem) for p in prompts])
        elapsed = time.perf_counter() - start
        arr = np.array(self.latencies)
        return {
            "total_requests": len(prompts),
            "successful": len(self.latencies),
            "errors": self.errors,
            "throughput_rps": len(self.latencies) / elapsed,
            "p50_ms": np.percentile(arr, 50) * 1000,
            "p95_ms": np.percentile(arr, 95) * 1000,
            "p99_ms": np.percentile(arr, 99) * 1000,
            "elapsed_s": round(elapsed, 2)
        }

# Run load test at different concurrency levels
deploy = LLMDeployment(max_batch_size=16)
test_prompts = [f"Generate a story about topic {i}" for i in range(50)]

print(f"{'Concurrency':>12} {'RPS':>8} {'P50ms':>8} {'P95ms':>8} {'P99ms':>8}")
print("-" * 50)
for conc in [1, 5, 10, 20]:
    client = LoadTestClient(deploy, concurrency=conc)
    stats = asyncio.get_event_loop().run_until_complete(client.run(test_prompts))
    print(f"{conc:>12} {stats['throughput_rps']:>8.1f} {stats['p50_ms']:>8.1f} "
          f"{stats['p95_ms']:>8.1f} {stats['p99_ms']:>8.1f}")

## 5. HAProxy Configuration for LLM Serving

Production HAProxy config with prefix-hash routing, health checks, and rate limiting.

In [ ]:
HAPROXY_CONFIG = """
global
    maxconn 4096
    log stdout format raw local0

defaults
    mode http
    timeout connect 5s
    timeout client  120s
    timeout server  120s
    option httplog
    option http-server-close

frontend llm_frontend
    bind *:8080
    # Rate limiting: 20 req/s per IP
    stick-table type ip size 100k expire 30s store http_req_rate(1s)
    http-request track-sc0 src
    http-request deny deny_status 429 if { sc_http_req_rate(0) gt 20 }
    # Route based on prompt prefix hash for KV cache affinity
    http-request set-header X-Prefix-Hash %[req.body,json_query('.prompt',''),bytes(0\,64),crc32]
    default_backend llm_replicas

backend llm_replicas
    balance hdr(X-Prefix-Hash)
    hash-type consistent
    option httpchk GET /health
    http-check expect status 200
    server ray-replica-0 10.0.1.10:8000 check inter 5s fall 3 rise 2 weight 100
    server ray-replica-1 10.0.1.11:8000 check inter 5s fall 3 rise 2 weight 100
    server ray-replica-2 10.0.1.12:8000 check inter 5s fall 3 rise 2 weight 100
    server ray-replica-3 10.0.1.13:8000 check inter 5s fall 3 rise 2 weight 100

listen stats
    bind *:9090
    stats enable
    stats uri /stats
    stats refresh 5s
"""

print("HAProxy LLM Serving Configuration:")
print("=" * 50)
print(HAPROXY_CONFIG)
print("\nKey features:")
print("  - Consistent hashing on prompt prefix (KV cache affinity)")
print("  - Rate limiting: 20 req/s per source IP")
print("  - Health checks every 5s, fail after 3 misses")
print("  - 120s timeouts for long generation requests")
print("  - Stats dashboard on :9090/stats")

In [ ]:
# Simulate consistent hashing behavior
import hashlib

def consistent_hash_route(prompt: str, num_replicas: int = 4) -> int:
    prefix = prompt[:64].encode()
    h = int(hashlib.md5(prefix).hexdigest(), 16)
    return h % num_replicas

# Verify prefix affinity
test_cases = [
    ("You are a helpful assistant. Q1", "You are a helpful assistant. Q2"),
    ("Summarize: doc_A", "Summarize: doc_B"),
    ("Translate to French: hello", "Translate to French: goodbye"),
]

print("Consistent hash routing verification:")
print(f"{'Prompt Pair':<45} {'Same Replica?':>13}")
print("-" * 60)
for p1, p2 in test_cases:
    r1, r2 = consistent_hash_route(p1), consistent_hash_route(p2)
    same = "✓ YES" if r1 == r2 else "✗ NO"
    print(f"{p1[:22]} / {p2[:18]:<18} {same:>13} (R{r1},R{r2})")

## Summary

| Component | Key Insight |
|-----------|-------------|
| Deployment Class | Encapsulates model + KV cache, async interface |
| Autoscaling | Aggressive profile reacts 12x faster but risks thrashing |
| Prefix Routing | 80%+ cache hit rate with shared system prompts |
| Load Testing | Concurrency reveals throughput ceiling and tail latency |
| HAProxy | Consistent hashing preserves prefix affinity at L7 |